[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C11_RAG_Retrieval_Course/06_long_context_eval/06_long_context_eval.ipynb)

# 06 · 长上下文评测（纯 numpy/pandas）

目标：把 **NIAH 大海捞针、lost-in-the-middle 位置偏置(U 形曲线)、NIAH 二维热图、多针指数衰减、RAG vs 长上下文、干扰项鲁棒性** 全部从零模拟出来，并用 `assert` 验证其定性规律。

路线：NIAH 构造 → U 形位置曲线 → NIAH 热图(长度×位置) → 多针连乘衰减 → RAG vs 长上下文 → 干扰项 → ✏️ 练习 → 📖 答案 → 🧪 真实文本 NIAH 胶囊。

> 心智模型：我们**不调用真实 LLM**，而是用一个**位置依赖的检索概率**（确定性 U 形函数）模拟模型对上下文不同位置信息的利用率，由此复现 NIAH 与 lost-in-the-middle 的典型形状。机制看懂了，换成真实模型的实测曲线，解读方式完全一致。

## 1 · NIAH 构造：把针插进草堆

NIAH = 把一句无关事实（**针 needle**）插进一大段无关文本（**草堆 haystack**），再问针对针的问题，看模型能否检索出。

先把『构造一个含针的上下文』这件事做出来：给定草堆 token 列表、针、以及**相对位置** `pos∈[0,1]`，把针插到对应处。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

def make_niah_context(haystack_tokens, needle, pos):
    '''把 needle 插进 haystack 的相对位置 pos∈[0,1] 处，返回(新上下文列表, 针的绝对下标)。'''
    n = len(haystack_tokens)
    insert_at = int(round(pos * n))           # pos=0 -> 开头, pos=1 -> 结尾
    insert_at = max(0, min(n, insert_at))
    ctx = haystack_tokens[:insert_at] + [needle] + haystack_tokens[insert_at:]
    return ctx, insert_at

haystack = [f'filler_{i}' for i in range(20)]   # 20 个无关 token
needle = 'THE_SECRET_IS_42'

for pos in [0.0, 0.5, 1.0]:
    ctx, idx = make_niah_context(haystack, needle, pos)
    print(f'pos={pos}: 针插在下标 {idx}/{len(ctx)-1}, 上下文长度 {len(ctx)}')
    assert ctx[idx] == needle, '针应在记录的下标处'
    assert len(ctx) == len(haystack) + 1, '插针后长度+1'
# pos=0 针在最前, pos=1 针在最后
ctx0, i0 = make_niah_context(haystack, needle, 0.0)
ctx1, i1 = make_niah_context(haystack, needle, 1.0)
assert i0 == 0 and ctx0[0] == needle
assert i1 == len(haystack) and ctx1[-1] == needle
print('✅ NIAH 上下文构造正确：针可插到任意相对位置')

## 2 · 位置偏置：U 形利用率曲线

**Lost in the Middle**：模型对**中段**信息的利用率显著低于**首尾**。我们用一个确定性 U 形函数模拟『模型读到位置 `r` 处针的利用率』：

$$u(r) = base + amp\cdot(2r-1)^2,\quad r\in[0,1]$$

`(2r-1)²` 在 `r=0.5` 处为 0（谷底）、在 `r=0,1` 处为 1（峰值）→ 两端高、中间低，正是 U 形。

In [ ]:
def position_utilization(r, base=0.45, amp=0.5):
    '''U 形利用率: 首尾高(~base+amp), 中段低(~base)。r 可为标量或数组。'''
    r = np.asarray(r, dtype=float)
    return base + amp * (2 * r - 1) ** 2

positions = np.linspace(0, 1, 11)
util = position_utilization(positions)
print('位置 :', np.round(positions, 1))
print('利用率:', np.round(util, 3))
mid = len(positions) // 2
# 画个文字版 U 形
for r, u in zip(positions, util):
    print(f'  r={r:.1f}  ' + '█' * int(u * 40) + f'  {u:.2f}')
assert util[0] > util[mid], '开头利用率应高于中段(primacy)'
assert util[-1] > util[mid], '结尾利用率应高于中段(recency)'
assert util[mid] == util.min(), '中段应是利用率谷底'
assert abs(util[0] - util[-1]) < 1e-9, '首尾对称'
print('✅ U 形位置曲线：首尾高、中段低 —— lost in the middle')

## 3 · NIAH 热图：长度 × 位置的二维扫描

真实 NIAH 同时扫 **上下文长度** 与 **针位置**。检索成功率 = `位置利用率 × 长度衰减`：
- 位置越靠中段越低（第 2 节的 U 形）；
- 上下文越长，整体利用率越低（**有效长度衰减**）。

我们生成这张二维热图(行=长度，列=位置)，并验证单调性：更长更难、中段更难。

In [ ]:
def length_decay(length, L_eff=8000.0):
    '''长度衰减: 上下文越长整体利用率越低(有效长度 L_eff 处衰减明显)。'''
    length = np.asarray(length, dtype=float)
    return 1.0 / (1.0 + length / L_eff)

def niah_success(length, pos, base=0.45, amp=0.5, L_eff=8000.0):
    '''NIAH 检索成功率 = 位置利用率 × 长度衰减。'''
    return position_utilization(pos, base, amp) * length_decay(length, L_eff)

lengths = [1000, 4000, 16000, 64000]
poss = [0.0, 0.25, 0.5, 0.75, 1.0]
heat = np.array([[niah_success(L, r) for r in poss] for L in lengths])
df = pd.DataFrame(np.round(heat, 3),
                  index=[f'{L//1000}k' for L in lengths],
                  columns=[f'{int(r*100)}%' for r in poss])
print('NIAH 成功率热图 (行=长度, 列=针位置):')
print(df.to_string())
# 单调性1: 越长越难(同一位置, 长度增 -> 成功率降)
for j in range(len(poss)):
    col = heat[:, j]
    assert np.all(np.diff(col) <= 1e-9), f'位置{poss[j]}: 越长应越难'
# 单调性2: 中段最难(每行的最小值在中间列)
for i in range(len(lengths)):
    assert heat[i].argmin() == 2, f'长度{lengths[i]}: 中段(50%)应最难'
# 角落(短+边缘) > 中心(长+中段)
assert heat[0, 0] > heat[-1, 2], '短上下文边缘 应远易于 长上下文中段'
print('✅ NIAH 热图：越长越难、中段最难 —— 右中部最红')

## 4 · 多针：成功率随针数指数衰减

**多针 multi-needle**：藏 `k` 根针，要求**全部**检索到。若单针成功率 `p`、各针近似独立，则全中概率 ≈ `p^k`——随针数**指数衰减**。一个单针 0.9 的模型，找 5 根针只剩 `0.9^5≈0.59`。

我们按位置给每根针算单针成功率，再连乘得全中概率，验证多针比单针难。

In [ ]:
def multi_needle_all_found(positions_list, length=16000):
    '''多根针分别在 positions_list 各位置, 返回(各针单针成功率, 全部找到的概率).'''
    per = np.array([niah_success(length, r) for r in positions_list])
    all_found = float(np.prod(per))
    return per, all_found

# 5 根针均匀分布在不同位置
five_pos = [0.1, 0.3, 0.5, 0.7, 0.9]
per, all5 = multi_needle_all_found(five_pos)
print('各针位置     :', five_pos)
print('各针单针成功率:', np.round(per, 3))
print(f'全部 5 针找到的概率 = {all5:.3f}')
# 单针(只取最好位置的那根)
single_best = per.max()
print(f'单针(最佳位置)成功率 = {single_best:.3f}')
assert all5 < single_best, '多针全中概率应远低于单针'
# 针越多越难: 前3针 vs 前5针
_, all3 = multi_needle_all_found(five_pos[:3])
assert all3 > all5, '针越多, 全中概率越低(连乘)'
# 指数衰减验证: 相同位置 p 的 k 根针 -> p^k
p = niah_success(16000, 0.5)
_, allk = multi_needle_all_found([0.5] * 4)
assert abs(allk - p ** 4) < 1e-9, '同位置 k 针应为 p^k'
print('✅ 多针：全中概率随针数指数衰减 —— 单针强不代表多针强')

## 5 · RAG vs 长上下文：把证据放对位置

关键洞察：**RAG 的优势不只是检索得全，还在于把最相关证据放在上下文最前(首位)**，吃到 primacy 高利用率；
而**长上下文直塞**常把关键信息埋在中段，撞上 U 形低谷。

同样的证据、同样的模型，仅仅因为**放的位置不同**，成功率就不同。我们量化这个差距。

In [ ]:
def long_context_stuff(evidence_pos, length=32000):
    '''长上下文直塞: 关键证据落在 evidence_pos(常在中段, 因为没排序).'''
    return niah_success(length, evidence_pos)

def rag_then_place(length_after_retrieval=2000):
    '''RAG: 检索后把最相关证据放首位(pos=0, 吃 primacy), 且上下文短得多.'''
    return niah_success(length_after_retrieval, 0.0)

# 一组问题, 长上下文直塞时证据散落在各位置(平均偏中段)
evidence_positions = [0.4, 0.5, 0.6, 0.45, 0.55]   # 没排序 -> 偏中段
lc_acc = np.mean([long_context_stuff(r) for r in evidence_positions])
rag_acc = np.mean([rag_then_place() for _ in evidence_positions])
print(f'长上下文直塞(证据落中段, 32k): 平均成功率 = {lc_acc:.3f}')
print(f'RAG(检索后证据放首位, 2k)   : 平均成功率 = {rag_acc:.3f}')
print(f'RAG 因把证据放对位置而获得的优势 = {rag_acc - lc_acc:+.3f}')
assert rag_acc > lc_acc, 'RAG 把证据放首位+缩短上下文 应优于长上下文直塞'
# 即便长上下文也把证据放首位, RAG 因上下文更短(长度衰减小)仍不吃亏
lc_front = np.mean([long_context_stuff(0.0) for _ in evidence_positions])
assert rag_acc >= lc_front - 1e-9, '同放首位, 短上下文(RAG)长度衰减更小'
print('✅ RAG 的隐性优势: 把同样的证据放到模型更用得上的位置(首位)+更短上下文')

## 6 · 干扰项：塞得多不等于塞得好

**干扰项 distractor**：表面相关、实则无关(或错误)的片段。干扰项越多，模型越可能被某个似是而非的骗过去，成功率下降。

我们建模：有真针时，成功率随干扰项数量上升而被『稀释』下降——这正是 context stuffing(不筛选直接塞满)的代价。

In [ ]:
def success_with_distractors(n_distractors, base_success=0.9, distract_strength=0.08):
    '''真针存在, 但每个干扰项以 distract_strength 概率把模型带偏; 成功率随干扰项数衰减。'''
    # 模型不被任何一个干扰项带偏的概率 ≈ (1-distract_strength)^n, 乘上基础成功率
    return base_success * (1 - distract_strength) ** n_distractors

print(f"{'干扰项数':>8s}{'成功率':>10s}")
prev = 1.1
for nd in [0, 2, 5, 10, 20]:
    s = success_with_distractors(nd)
    print(f'{nd:>8d}{s:>10.3f}')
    assert s <= prev, '干扰项越多, 成功率应单调不升'
    prev = s
# 无干扰项 = 基础成功率
assert abs(success_with_distractors(0) - 0.9) < 1e-9
# 大量干扰项显著拉低
assert success_with_distractors(20) < 0.5 * success_with_distractors(0)
print('✅ 干扰项: 塞得越多越易被带偏 —— context stuffing 的代价, RAG 用高 precision 检索规避')

---
## ✏️ 练习 1：NIAH 上下文构造

实现 `insert_needle(haystack, needle, pos)`：把 `needle` 插到 `haystack`(列表)的相对位置 `pos∈[0,1]` 处，返回 `(新列表, 针的绝对下标)`。`pos=0` 插最前，`pos=1` 插最后。

In [ ]:
def insert_needle(haystack, needle, pos):
    # TODO: insert_at = round(pos*len(haystack)), 夹到[0,len]; 在该处插入 needle
    #       返回 (新列表, insert_at)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
hs = list('ABCDEFGHIJ')
ctx, idx = insert_needle(hs, 'N', 0.5)
assert ctx[idx] == 'N' and len(ctx) == 11
c0, i0 = insert_needle(hs, 'N', 0.0)
c1, i1 = insert_needle(hs, 'N', 1.0)
assert i0 == 0 and c0[0] == 'N', 'pos=0 应在最前'
assert i1 == 10 and c1[-1] == 'N', 'pos=1 应在最后'
assert insert_needle(hs, 'N', 0.5)[1] == 5, 'pos=0.5 应在正中'
print('✅ 练习 1 通过：NIAH 插针正确')

## ✏️ 练习 2：U 形位置偏置曲线

实现 `u_curve(positions, base=0.4, amp=0.5)`：返回各位置的 U 形利用率数组(首尾高、中段低)，并实现 `is_u_shaped(curve)` 判断一条曲线是否 U 形(两端都大于中间最小值附近)。

In [ ]:
def u_curve(positions, base=0.4, amp=0.5):
    # TODO: 返回 base + amp*(2*positions-1)**2
    raise NotImplementedError

def is_u_shaped(curve):
    # TODO: 两端值 > 最小值, 且最小值出现在中间区域(非两端) -> True
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
pos = np.linspace(0, 1, 9)
cur = u_curve(pos)
assert cur[0] > cur[len(cur)//2] and cur[-1] > cur[len(cur)//2]
assert is_u_shaped(cur) == True, '该曲线应被判为 U 形'
# 单调递增曲线不是 U 形
assert is_u_shaped(np.array([0.1, 0.2, 0.3, 0.4, 0.5])) == False
# 谷底在中间
assert cur.argmin() == len(cur)//2, '谷底应在正中'
print('✅ 练习 2 通过：U 形曲线构造与判定正确')

## ✏️ 练习 3：多针指数衰减

实现 `multi_needle_success(per_needle_probs)`：给定各针的单针成功率列表，返回**全部找到**的概率(连乘)。再实现 `needles_for_target(p, target)`：单针成功率 `p`、要求全中概率 ≥ `target`，最多能放几根针？

In [ ]:
def multi_needle_success(per_needle_probs):
    # TODO: 返回所有单针成功率的连乘
    raise NotImplementedError

def needles_for_target(p, target):
    # TODO: 最大的 k 使 p**k >= target (p<1); 返回 int (k>=0)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(multi_needle_success([0.9, 0.9, 0.9]) - 0.729) < 1e-9
assert multi_needle_success([1.0, 1.0]) == 1.0
# 单针 0.9, 要全中>=0.5: 0.9^k>=0.5 -> k<=6 (0.9^6=0.531, 0.9^7=0.478)
assert needles_for_target(0.9, 0.5) == 6, '0.9^6=0.53>=0.5, 0.9^7<0.5'
# 单针越强能放越多针
assert needles_for_target(0.99, 0.5) > needles_for_target(0.9, 0.5)
print('✅ 练习 3 通过：多针连乘与容量计算正确')

## ✏️ 练习 4：RAG vs 长上下文

用第 3 节的 `niah_success`。实现 `rag_vs_lc(evidence_pos, lc_length, rag_length)`：返回 `(长上下文直塞成功率, RAG放首位成功率, RAG优势)`。长上下文证据在 `evidence_pos`、长度 `lc_length`；RAG 证据放 `pos=0`、长度 `rag_length`。验证证据在中段时 RAG 更优。

In [ ]:
def rag_vs_lc(evidence_pos, lc_length=32000, rag_length=2000):
    # TODO: lc = niah_success(lc_length, evidence_pos)
    #       rag = niah_success(rag_length, 0.0)
    #       返回 (lc, rag, rag-lc)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
lc, rag, adv = rag_vs_lc(0.5)         # 证据在正中段
assert rag > lc and adv > 0, '证据在中段时 RAG(放首位)应更优'
# 证据若本就在首位, 长上下文劣势仅来自长度衰减
lc2, rag2, adv2 = rag_vs_lc(0.0)
assert adv2 >= 0, 'RAG 上下文更短, 长度衰减更小, 不吃亏'
# 证据越靠中段, RAG 优势越大
assert rag_vs_lc(0.5)[2] > rag_vs_lc(0.1)[2]
print(f'证据在中段: 长上下文={lc:.3f}, RAG={rag:.3f}, 优势={adv:+.3f}')
print('✅ 练习 4 通过：量化了 RAG 把证据放对位置的优势')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def insert_needle(haystack, needle, pos):
    n = len(haystack)
    insert_at = int(round(pos * n))
    insert_at = max(0, min(n, insert_at))
    return haystack[:insert_at] + [needle] + haystack[insert_at:], insert_at

In [ ]:
# 练习 2 参考答案
def u_curve(positions, base=0.4, amp=0.5):
    positions = np.asarray(positions, dtype=float)
    return base + amp * (2 * positions - 1) ** 2

def is_u_shaped(curve):
    curve = np.asarray(curve, dtype=float)
    i_min = curve.argmin()
    return bool(0 < i_min < len(curve) - 1 and curve[0] > curve[i_min] and curve[-1] > curve[i_min])

In [ ]:
# 练习 3 参考答案
def multi_needle_success(per_needle_probs):
    return float(np.prod(per_needle_probs))

def needles_for_target(p, target):
    k = 0
    while p ** (k + 1) >= target:
        k += 1
    return k

In [ ]:
# 练习 4 参考答案
def rag_vs_lc(evidence_pos, lc_length=32000, rag_length=2000):
    lc = niah_success(lc_length, evidence_pos)
    rag = niah_success(rag_length, 0.0)
    return lc, rag, rag - lc

---
## 🧪 真实数据胶囊：在真实文本上跑 lost-in-the-middle

用**真实的长文本**当草堆：我们优先**联网**拉取一段真实公开文本(维基/古登堡)，失败则**回退到内置的真实文本段落**。

把一句真实事实(针)插到这段真实文本的不同位置，用我们的位置依赖模型算检索成功率，**在真实文本上复现 U 形曲线**——草堆是真的，只有『模型』是我们模拟的位置检索器。

In [ ]:
def load_real_haystack():
    '''优先联网拉真实公开文本; 失败回退内置真实文本(逐字摘自维基百科 'Information retrieval' 词条).'''
    try:
        import urllib.request
        # 维基百科 REST API 取一篇文章摘要(真实文本)
        url = 'https://en.wikipedia.org/api/rest_v1/page/summary/Information_retrieval'
        import json
        with urllib.request.urlopen(url, timeout=5) as f:
            data = json.load(f)
        text = data.get('extract', '')
        if len(text.split()) < 20:
            raise RuntimeError('extract too short')
        print(f'✅ 联网加载真实文本成功 ({len(text.split())} 词)')
        return text
    except Exception as e:
        print(f'⚠ 联网失败({type(e).__name__})，回退内置真实文本')
        # 逐字摘自维基百科 'Information retrieval' 词条的真实文本
        return ('Information retrieval is the science of searching for information '
                'in a document, searching for documents themselves, and also searching '
                'for the metadata that describes data, and for databases of texts, images '
                'or sounds. Automated information retrieval systems are used to reduce what '
                'has been called information overload. An IR system is a software system that '
                'provides access to books, journals and other documents; it also stores and '
                'manages those documents. Web search engines are the most visible IR applications.')

text = load_real_haystack()
haystack_tokens = text.split()
needle = 'REMEMBER_THE_MAGIC_NUMBER_IS_1729'
print(f'真实草堆: {len(haystack_tokens)} 个词; 针 = {needle!r}')
# 把针插到不同位置, 算成功率(草堆真实, 模型=位置检索器)
for pos in [0.0, 0.5, 1.0]:
    ctx, idx = make_niah_context(haystack_tokens, needle, pos)
    s = niah_success(len(ctx), pos)
    print(f'  针在 {int(pos*100):>3d}% (下标 {idx:>3d}): 成功率 {s:.3f}')

**🧪 胶囊练习**：实现 `lost_in_middle_on_text(haystack_tokens, needle, n_positions=11)`：把针扫过 `n_positions` 个相对位置，返回 `(positions数组, 成功率数组)`，并应满足首尾成功率 > 中段(在真实文本上复现 U 形)。

In [ ]:
def lost_in_middle_on_text(haystack_tokens, needle, n_positions=11):
    # TODO: positions = linspace(0,1,n_positions)
    #       对每个 pos: 构造含针上下文, 用 niah_success(len(ctx), pos) 算成功率
    #       返回 (positions, successes)
    raise NotImplementedError

In [ ]:
# 自测
positions, succ = lost_in_middle_on_text(haystack_tokens, needle)
mid = len(positions) // 2
print('位置 :', np.round(positions, 2))
print('成功率:', np.round(succ, 3))
assert len(positions) == len(succ) == 11
assert succ[0] > succ[mid] and succ[-1] > succ[mid], '真实文本上也应是 U 形(首尾>中段)'
print('✅ 胶囊练习通过：在真实文本上复现了 lost-in-the-middle 的 U 形曲线')

In [ ]:
# 📖 胶囊参考答案
def lost_in_middle_on_text(haystack_tokens, needle, n_positions=11):
    positions = np.linspace(0, 1, n_positions)
    successes = []
    for pos in positions:
        ctx, _ = make_niah_context(haystack_tokens, needle, pos)
        successes.append(niah_success(len(ctx), pos))
    return positions, np.array(successes)

### 小结
- **NIAH 大海捞针**：针插进草堆、扫(长度×位置)二维热图——长上下文事实检索的事实标准；**必要但不充分**。
- **lost in the middle**(Liu 2023)：利用率 U 形，首尾高中段低；启示——RAG 把**最相关文档放首尾**(别埋中段)。
- **位置曲线**：首尾强中段弱源于注意力稀释+训练分布+位置外推；与**有效长度衰减**是两个交织的轴。
- **多针/多跳**：全中概率 `p^k` 指数衰减；RULER(2024) 揭示**有效长度远短于宣称**(『支持128k』≠『128k 处仍准』)。
- **干扰项 / context stuffing**：塞得多≠塞得好，干扰项越多越易被带偏；RAG 用高 precision 检索规避。
- **RAG vs 长上下文**：互补而非替代(Xu 2023)；RAG 省 token/可溯源/可更新/规避位置偏置，长窗口简单/容量大。

---
## 🎓 全课收束

你已经把一条**完整的检索增强全栈**从零搭了出来并学会量化每一环：

**01 嵌入** → **02 向量检索** → **03 重排** → **04 RAG 评测** → **05 检索指标** → **06 长上下文评测**

贯穿始终的一条主线是：**把对的知识，以可度量、可优化的方式，放到模型最用得上的地方**。
- 检索(01–02)回答『**哪些**信息最相关』；重排(03)回答『**精挑**出哪几条』；
- 评测(04–05)回答『检索/生成**好不好、坏在哪**』；长上下文(06)回答『窗口能装多少、该把证据放**哪个位置**』。

换上真实 embedding、真实向量库(FAISS)、真实重排器(monoBERT)、真实评测(RAGAS)、真实长上下文模型——本课的每一行结构与度量代码，**逻辑都一字不用改**。这正是『从零搭全栈』给你的底气：不被工具绑架，能对任何 RAG 系统做出可量化、可复现、能定位瓶颈的判断。

🎉 **恭喜你完成《检索增强与长上下文评测》全部 7 个模块！**